# Lab 1 : From a handbook to a search

*W3 RAG Part 1 · Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.


## What we are achieving in this lab

A chat model does not look up your company handbook. **RAG** is the usual fix: cut the file into chunks, turn each chunk into numbers, find the closest chunks to a question, then (next lab) let the model answer only from those chunks.

```
load → split → embed → store → retrieve → generate
 this lab                              next lab
```

**Prerequisites.** Weeks 1 and 2 finished. Copy `.env.example` to `.env` at the repository root and paste `OPENAI_API_KEY` before Step 4. See [README.md](./README.md).

**What you will do, in this order.**

1. **Chunk** the handbook — why size matters.
2. **Embed** a piece of text — it becomes a list of numbers.
3. **Cosine** — one score that means "close in meaning."
4. **Keyword vs cosine** — words vs meaning, same question.
5. **Vector store** — the same search, without the `for` loop.

**Cost.** A handful of embedding calls. Fractions of a cent.


### Step 1. Load the handbook

`handbook.txt` sits next to this notebook. Print it once so you know what you are cutting.


In [1]:
with open("handbook.txt", encoding="utf-8") as f:
    HANDBOOK = f.read()

print("characters:", len(HANDBOOK))
print()
print(HANDBOOK)


file      : handbook.txt
characters: 1399

# Onboarding Handbook

## Section 1: Setting Up Your Account
To set up your account, visit the company portal at portal.example.com.
Click the "Sign Up" button. You will receive a confirmation email within
five minutes. If you do not see the email, check your spam folder.
The portal supports two factor authentication, which we strongly recommend.

## Section 2: Your First Week
In your first week, your manager will walk you through three things:
the team rituals, the on-call calendar, and the deployment pipeline.
You will also meet your buddy, who is your primary point of contact for
non-urgent questions for your first 30 days.

## Section 3: Reimbursements
To submit a reimbursement, log into the finance portal at finance.example.com.
Upload your receipt as a PDF. Reimbursements take 7 to 10 business days.
For travel under $500, no pre-approval is needed. Above $500 requires
your manager and finance team approval.

## Section 4: Time Off
Submit

### Step 2. One file, several topics

This handbook covers more than one topic: account setup, reimbursements, time off, and parental leave. If you embed the whole file as a single vector, those topics are combined into one set of numbers. A question about a train ticket then has to match against that combined vector, which is less accurate.

The next cell is not the real splitter. It only splits on the `##` headings so you can see each topic and how long it is. Step 3 is where we cut the file into chunks we can embed.


In [2]:
# Split only where the file has a markdown heading (## ).
# This is a quick look at the topics. It is not how we chunk for RAG.
parts = HANDBOOK.split("\n## ")

print("sections:", len(parts))
print()
for part in parts:
    heading = part.split("\n", 1)[0]
    heading = heading.replace("#", "").strip()
    print(len(part), "chars  ", heading)


sections: 6

22 chars   Onboarding Handbook
323 chars   Section 1: Setting Up Your Account
282 chars   Section 2: Your First Week
288 chars   Section 3: Reimbursements
227 chars   Section 4: Time Off
237 chars   Section 5: Parental Leave


### Step 3. Cut the file into chunks

Step 2 listed headings. It did not create the chunks we will embed.

A real document may have no headings, or one section that is too long. So we cut by length, using LangChain's `RecursiveCharacterTextSplitter`. Read the comments in the next cell.

We try three sizes on the same file: **200** (too small), **500** (usable), **2000** (larger than this file, so still one piece).


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter cuts a long string into smaller strings (chunks).
# It tries to cut at a paragraph break first, then a newline, then a space,
# so it does not split in the middle of a word.
#
# chunk_size    = maximum number of characters in one chunk
# chunk_overlap = how many characters at the end of one chunk are copied
#                 onto the start of the next chunk
#
# Example, overlap = 0 (cut is a hard edge):
#   text:    Pay the receipt. Above 500 needs approval.
#   chunk 1: Pay the receipt. Above
#   chunk 2:                  500 needs approval.
#   A search that finds chunk 1 never sees "needs approval".
#
# Same text, overlap > 0 (the end of chunk 1 starts chunk 2):
#   chunk 1: Pay the receipt. Above 500
#   chunk 2:                  Above 500 needs approval.
#   "Above 500" sits in both chunks, so the rule is not lost on the cut.


def split_handbook(chunk_size: int, chunk_overlap: int) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    # Returns a list of strings. Each string is one chunk of HANDBOOK.
    return splitter.split_text(HANDBOOK)


def show_chunks(label: str, chunks: list[str], print_all: bool) -> None:
    print("=====", label, "->", len(chunks), "chunks =====")
    print()
    if print_all:
        for i, text in enumerate(chunks):
            print("--- chunk", i, "(", len(text), "chars) ---")
            print(text)
            print()
        return
    
    print("--- first chunk ---")
    print(chunks[0])
    print()
    print("--- last chunk ---")
    print(chunks[-1])
    print()


# Same file, three sizes. Compare the printed chunks.
small = split_handbook(200, 40)    # too small: a rule can be split across two chunks
medium = split_handbook(500, 50)   # usable for this handbook; we keep this list
huge = split_handbook(2000, 0)     # larger than the file, so this is still one chunk

show_chunks("too small (200)", small, print_all=False)
show_chunks("usable (500)", medium, print_all=True)
show_chunks("whole file (2000)", huge, print_all=False)


===== too small (200) -> 11 chunks =====

--- first chunk ---
# Onboarding Handbook

--- last chunk ---
Notify HR at least 30 days in advance when the date is predictable.

===== usable (500) -> 4 chunks =====

--- chunk 0 ( 348 chars) ---
# Onboarding Handbook

## Section 1: Setting Up Your Account
To set up your account, visit the company portal at portal.example.com.
Click the "Sign Up" button. You will receive a confirmation email within
five minutes. If you do not see the email, check your spam folder.
The portal supports two factor authentication, which we strongly recommend.

--- chunk 1 ( 284 chars) ---
## Section 2: Your First Week
In your first week, your manager will walk you through three things:
the team rituals, the on-call calendar, and the deployment pipeline.
You will also meet your buddy, who is your primary point of contact for
non-urgent questions for your first 30 days.

--- chunk 2 ( 290 chars) ---
## Section 3: Reimbursements
To submit a reimbursement, log into t

Read that output.

- **200:** "Upload your receipt" can land in one chunk and "Above $500 requires approval" in the next. A $300 ticket needs both sentences.
- **500:** A section usually fits. We keep `medium` for the rest of this lab.
- **2000:** One chunk. You are back to the whole handbook.

The rest of this lab uses **500**. Too small loses a rule. Too big mixes every topic.


### Step 4. Load the OpenAI key

From here on we call a hosted API. The text you embed leaves your laptop.

Copy `.env.example` to `.env` at the repository root. Paste `OPENAI_API_KEY`. Never commit `.env`.


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("Set OPENAI_API_KEY in the repo-root .env file.")

print("OPENAI_API_KEY : set")


OPENAI_API_KEY : set


### Step 5. An embedding is a list of numbers

We use OpenAI `text-embedding-3-small` through LangChain. It is cheap (~$0.02 / 1M tokens) and current.

**Anthropic does not offer an embedding model.** Claude writes answers. It does not turn text into vectors. A Claude RAG app still needs an embedder (OpenAI, Voyage, Cohere, …).

One rule: **one index, one embedding model.** If you change the model, you must embed every chunk again.


In [5]:
from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

text = "How do I reset my password?"
vec = embeddings.embed_query(text)

print("text   :", text)
print("model  :", EMBED_MODEL)
print("type   :", type(vec).__name__)
print("length :", len(vec), "numbers")
print("first 8:", [round(x, 4) for x in vec[:8]])


text   : How do I reset my password?
model  : text-embedding-3-small
type   : list
length : 1536 numbers
first 8: [0.0176, -0.0457, 0.0298, 0.0219, -0.0486, 0.002, 0.0016, 0.0614]


There is no hidden object. An embedding is a Python list of floats. For this model the list is always **1536** long — a word or a paragraph, same length. That shared length is what lets you compare them.


In [6]:
short = "password"
long = (
    "To reset your password, open the sign-in page, click Forgot Password, "
    "and check the email we send within five minutes."
)

short_vec = embeddings.embed_query(short)
long_vec = embeddings.embed_query(long)

print(repr(short), "->", len(short_vec), "numbers")
print("paragraph   ->", len(long_vec), "numbers")
print("same length :", len(short_vec) == len(long_vec))


'password' -> 1536 numbers
paragraph   -> 1536 numbers
same length : True


RAG uses two calls:

| When | Method | Input |
|------|--------|--------|
| You build the index (chunks change) | `embed_documents` | many strings |
| A user asks a question | `embed_query` | one string |


In [7]:
query_vec = embeddings.embed_query(text)
doc_vecs = embeddings.embed_documents(
    [
        "Click Forgot Password on the sign-in page.",
        "Our offices open at 9am Pacific.",
    ]
)

print("embed_query    : 1 vector of", len(query_vec), "numbers")
print("embed_documents:", len(doc_vecs), "vectors, each", len(doc_vecs[0]), "numbers")


embed_query    : 1 vector of 1536 numbers
embed_documents: 2 vectors, each 1536 numbers


### Step 6. Cosine: how close are two vectors?

Step 5 gave you two lists of 1536 numbers. You cannot tell by eye whether they are close. You need one score.

**Cosine similarity** is that score. Treat each embedding as an arrow. Cosine asks: do these two arrows point in the same direction?

| Score | Meaning |
|-------|---------|
| **1.0** | Same direction. The two texts are identical, or very close in meaning. |
| **near 0** | Almost a right angle. The texts are unrelated. |
| **near -1** | Opposite directions. Rare for these text models. |

For this course, **higher cosine means closer in meaning**. Search ranks chunks by this number and keeps the highest.

**How the formula works** (you do not need to derive it):

1. Line the two lists up, slot by slot (both must be length 1536, same model).
2. Multiply each pair of numbers and add those products. That is the **dot product**. It is large when the arrows point the same way.
3. Divide by the **length** of each list. A long paragraph would otherwise score high just because the list has large values, not because the meaning matches.

The next cell writes that in three lines, then checks a required fact: a vector compared with itself must be `1.0`. If it is not, the formula is wrong or the two vectors did not come from the same model.


In [8]:
import numpy as np


def embed(text: str) -> np.ndarray:
    # embed_query returns a normal Python list of 1536 floats.
    # asarray turns that list into a NumPy array so we can use
    # np.dot and np.linalg.norm in cosine() below.
    # dtype=float means store each number as a decimal, which the math needs.
    return np.asarray(embeddings.embed_query(text), dtype=float)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    # np.dot(a, b)           = multiply slot by slot, then add (dot product)
    # np.linalg.norm(a)      = length of vector a
    # divide by both lengths = long text does not win just because it is long
    # result                 = 1.0 if the arrows point the same way
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


hello = embed("hello")
print("dimension :", hello.shape[0], "  (must stay 1536 for this model)")
print("cosine of a vector with itself :", round(cosine(hello, hello), 3), "  (must be 1.0)")




dimension : 1536   (must stay 1536 for this model)
cosine of a vector with itself : 1.0   (must be 1.0)


If that last number is not `1.0`, stop. The two vectors must come from the **same** model.

Now rank a few sentences against one question. Read the **order**, not the third decimal.


In [9]:
question = "How do I reset my password?"
q_vec = embed(question)

candidates = [
    "How do I reset my password?",
    "I forgot my password, can you help?",
    "How do I change my login credentials?",
    "What time does the office open?",
    "How do I cook pasta?",
]

print("Question:", question)
print()

scored = []
for text in candidates:
    score = cosine(q_vec, embed(text))
    scored.append((score, text))
    print(round(score, 3), " ", text)

print()
print("Highest first:")
scored.sort(reverse=True)
for score, text in scored:
    print(round(score, 3), " ", text)


Question: How do I reset my password?

1.0   How do I reset my password?
0.637   I forgot my password, can you help?
0.652   How do I change my login credentials?
0.142   What time does the office open?
0.216   How do I cook pasta?

Highest first:
1.0   How do I reset my password?
0.652   How do I change my login credentials?
0.637   I forgot my password, can you help?
0.216   How do I cook pasta?
0.142   What time does the office open?


Same meaning stays at the top even when the words change ("credentials"). Pasta and office hours drop. That drop is why embedding search works.


### Step 7. Keyword search vs embedding search

Cosine is not the only way to search. **Keyword search** counts shared words. No vectors. No API.

The snippets below say **"credentials"**. We ask about **"sign-in details"**. Same meaning, different words.


In [10]:
CORPUS = [
    "To update your credentials, visit Account > Profile > Edit.",
    "Our offices open at 9am Pacific.",
    "If you forgot your password, click Forgot Password on the sign-in page.",
    "We accept Visa, Mastercard, and American Express.",
    "To change your email address, open Account > Security.",
    "Refunds are issued within 30 days under our standard policy.",
]

QUERY_PARAPHRASE = "Where can users edit their sign-in details?"


def words(text: str) -> list[str]:
    cleaned = text.lower().replace(",", "").replace(".", "").replace("?", "")
    return cleaned.split()


query_words = words(QUERY_PARAPHRASE)
print("Question:", QUERY_PARAPHRASE)
print("Words   :", query_words)
print()
print("--- keyword (shared words) ---")
for doc in CORPUS:
    shared = []
    for w in query_words:
        if w in words(doc) and w not in shared:
            shared.append(w)
    print("matches =", len(shared), " ", shared if shared else "-", " ", doc)


Question: Where can users edit their sign-in details?
Words   : ['where', 'can', 'users', 'edit', 'their', 'sign-in', 'details']

--- keyword (shared words) ---
matches = 1   ['edit']   To update your credentials, visit Account > Profile > Edit.
matches = 0   -   Our offices open at 9am Pacific.
matches = 1   ['sign-in']   If you forgot your password, click Forgot Password on the sign-in page.
matches = 0   -   We accept Visa, Mastercard, and American Express.
matches = 0   -   To change your email address, open Account > Security.
matches = 0   -   Refunds are issued within 30 days under our standard policy.


The useful row ("credentials") should show **0** or a leftover like `"edit"`. Keywords cannot see a paraphrase.

Same question, cosine:


In [11]:
print("Question:", QUERY_PARAPHRASE)
print()
print("--- cosine (meaning) ---")

q_vec = embed(QUERY_PARAPHRASE)
scored = []
for doc in CORPUS:
    score = cosine(q_vec, embed(doc))
    scored.append((score, doc))
    print(round(score, 3), " ", doc)

print()
print("Highest first:")
scored.sort(reverse=True)
for score, doc in scored:
    print(round(score, 3), " ", doc)


Question: Where can users edit their sign-in details?

--- cosine (meaning) ---
0.551   To update your credentials, visit Account > Profile > Edit.
0.143   Our offices open at 9am Pacific.
0.423   If you forgot your password, click Forgot Password on the sign-in page.
0.183   We accept Visa, Mastercard, and American Express.
0.458   To change your email address, open Account > Security.
0.128   Refunds are issued within 30 days under our standard policy.

Highest first:
0.551   To update your credentials, visit Account > Profile > Edit.
0.458   To change your email address, open Account > Security.
0.423   If you forgot your password, click Forgot Password on the sign-in page.
0.183   We accept Visa, Mastercard, and American Express.
0.143   Our offices open at 9am Pacific.
0.128   Refunds are issued within 30 days under our standard policy.


Meaning search should put the credentials row first.

Keep both ideas:

- **Keywords** miss paraphrases. They catch an exact token (error code, SKU). Production name: **BM25**.
- **Cosine** finds meaning. It can bury a SKU in a long lunch paragraph.
- Production RAG often runs **both** (hybrid). Week 4 builds that. Not today.


### Step 8. Search the handbook chunks you already made

Same cosine, now on `medium` from Step 3. This is the whole point of chunking: each chunk is one topic, so a train-ticket question can land on reimbursements instead of the whole mixed file.


In [12]:
QUERY = "How do I get reimbursed for a $300 train ticket?"
print("Question:", QUERY)
print("chunks :", len(medium), "  (the 500-size split from Step 3)")
print()

chunk_vecs = embeddings.embed_documents(medium)
q_vec = embed(QUERY)

scored = []
for i, text in enumerate(medium):
    score = cosine(q_vec, np.asarray(chunk_vecs[i], dtype=float))
    scored.append((score, i, text))
    preview = text.replace("\n", " ")
    if len(preview) > 80:
        preview = preview[:80] + "..."
    print(round(score, 3), "  chunk", i, " ", preview)

print()
scored.sort(reverse=True)
best_score, best_i, best_text = scored[0]
print("Highest: chunk", best_i, "  cosine", round(best_score, 3))
print()
print(best_text)


Question: How do I get reimbursed for a $300 train ticket?
chunks : 4   (the 500-size split from Step 3)

0.095   chunk 0   # Onboarding Handbook  ## Section 1: Setting Up Your Account To set up your acco...
0.081   chunk 1   ## Section 2: Your First Week In your first week, your manager will walk you thr...
0.527   chunk 2   ## Section 3: Reimbursements To submit a reimbursement, log into the finance por...
0.18   chunk 3   ## Section 4: Time Off Submit time off requests through the HR portal. We have a...

Highest: chunk 2   cosine 0.527

## Section 3: Reimbursements
To submit a reimbursement, log into the finance portal at finance.example.com.
Upload your receipt as a PDF. Reimbursements take 7 to 10 business days.
For travel under $500, no pre-approval is needed. Above $500 requires
your manager and finance team approval.


The top chunk should be Section 3 (reimbursements), including the $500 line. That is chunking + embedding + cosine working together.


### Step 9. A vector store does that loop for you

A product will not score every chunk in Python on every question. A **vector store** keeps the vectors and returns the closest few.

`InMemoryVectorStore` is the lab version: it lives in this kernel only. Restart, and it is empty. Production names you should recognise: **Chroma** (files on disk), **pgvector** (Postgres), **Pinecone / Qdrant** (hosted). The LangChain line you write stays `retriever.invoke(question)`.


In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

docs = []
for i, text in enumerate(medium):
    docs.append(Document(page_content=text, metadata={"chunk": i}))

vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("stored", len(docs), "chunks")
print("Question:", QUERY)
print()

hits = retriever.invoke(QUERY)
for i, hit in enumerate(hits, start=1):
    print("--- hit", i, "  chunk", hit.metadata.get("chunk"), " ---")
    print(hit.page_content)
    print()


Hit 1 should be the same reimbursement chunk your cosine loop put first. The store did not invent a new score. It ran embed + cosine for you.

- **`from_documents`** = index time (when the handbook changes).
- **`retriever.invoke`** = question time (every search).


## What you should be able to explain

> "I split a document into chunks before I embed. Too small loses a rule. Too big mixes every topic."

> "An embedding is a fixed-length list of numbers. I use one model per index. Claude does not embed."

> "Cosine is how I score close. Higher means closer in meaning. Keyword search matches words; cosine matches meaning."

> "A vector store keeps those chunk vectors and returns the closest few. Under the hood that is still embed + cosine."

**Lab 2** takes these hits and asks a chat model to answer **only** from them.
